# Nevada conventional demonstration: 150 °C at 3 km

This notebook uses only public geoPFA interfaces and an explicit configuration cell. Local source data and generated outputs are deliberately not version-controlled. The fitted GBLK model predicts the published Great Basin proxy-positive endpoint; a separate Stanford thermal-model panel reports `P(T(3 km) > 150 °C)`. Keeping these quantities separate avoids treating proxy labels as measured temperature outcomes.

In [ ]:
import copy
import os
import pickle
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from scipy.stats import norm

from geopfa.layer_combination import VoterVeto
from geopfa.prob import ProbabilisticConfig, run_probabilistic
from geopfa.prob.gblk_runner import run_gblk_calibration_cv

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise FileNotFoundError("geoPFA repository root not found")


def required_local_path(env_name: str, default: Path) -> Path:
    path = Path(os.environ.get(env_name, default)).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(
            f"Required local input is absent: {path}. Set {env_name} or prepare "
            "the documented public study data before running this notebook."
        )
    return path

repo_root = find_repo_root(Path.cwd())
project_dir = repo_root / "examples" / "Nevada" / "2D"
output_root = Path(
    os.environ.get("GEOPFA_DEMO_OUTPUT_ROOT", project_dir / "outputs")
).expanduser().resolve()
output_dir = output_root / "nevada_conventional_150c_3km"
pfa_path = required_local_path(
    "GEOPFA_NEVADA_PFA", project_dir / "notebooks" / "fpa.pkl"
)
labels_path = required_local_path(
    "GEOPFA_NEVADA_LABELS",
    repo_root / "data" / "raw" / "nevada" / "labeled_wells_nevada.gpkg",
)
thermal_mean_path = required_local_path(
    "GEOPFA_NEVADA_3KM_MEAN",
    project_dir / "data" / "thermal" / "stanford_temperature_3km_mean_c.tif",
)
thermal_sd_path = required_local_path(
    "GEOPFA_NEVADA_3KM_SD",
    project_dir / "data" / "thermal" / "stanford_temperature_3km_sd_c.tif",
)
with pfa_path.open("rb") as stream:
    pfa = pickle.load(stream)

config_dict = {'enabled': True,
 'output_dir': '../outputs/conventional_150c_3km',
 'dimensions': '2d',
 'labels': {'source': '../../../../data/labeled_wells/nevada_great_basin/labeled_wells_nevada.gpkg',
            'id_col': 'well_id',
            'label_columns': {'heat': 'heat_label'},
            'label_quality_col': 'label_quality',
            'label_source_col': 'label_source',
            'pu_mode': 'off',
            'min_wells_for_fit': 5},
 'alpha': {'heat': {'mode': 'scalar',
                    'scalar_fallback_pr0': 0.57,
                    'force_prior_predictive': False,
                    'use_evidence_prior': False}},
 'evidence': {'regularization': {'C': 0.1}},
 'spatial_field': {'enabled': True,
                   'backend': 'latticekrigx',
                   'n_levels': 2,
                   'lattice_centers_per_dimension': 3,
                   'coordinate_scaling': 'axis_range'},
 'inference': {'backend': 'gblk', 'gblk_bayesian': {'enabled': False}},
 'calibration': {'method': 'none', 'fit_on': 'block_cv'},
 'cross_validation': {'n_folds': 5,
                      'block_type': 'grid',
                      'block_size_km': 20.0,
                      'buffer_km': 10.0},
 'combination': {'rule': 'product'},
 'scenarios': [],
 'outputs': {'probability_rasters': True,
             'uncertainty_rasters': False,
             'format': ['geotiff', 'csv']}}
config_dict["output_dir"] = str(output_dir)
config_dict["labels"]["source"] = str(labels_path)
config = ProbabilisticConfig.from_dict(config_dict)
config

In [ ]:
pfa_vv = VoterVeto.do_voter_veto(
    copy.deepcopy(pfa),
    normalize_method="minmax",
    component_veto=False,
    criteria_veto=True,
    normalize=True,
    norm_to=5,
)
model_result = run_probabilistic(
    pfa,
    config,
    input_artifacts={"pfa_pickle": pfa_path, "labels": labels_path},
)
probability = model_result.components["heat"].probability.copy()
probability["x"] = probability.geometry.x
probability["y"] = probability.geometry.y
with rasterio.open(thermal_mean_path) as source:
    temperature_mean = source.read(1)
    thermal_extent = [source.bounds.left, source.bounds.right, source.bounds.bottom, source.bounds.top]
with rasterio.open(thermal_sd_path) as source:
    temperature_sd = source.read(1)
thermal_probability = norm.sf((150.0 - temperature_mean) / temperature_sd)
probability[["probability"]].describe()

In [ ]:
vv = pfa_vv["criteria"]["geologic"]["components"]["heat"]["pr_norm"].copy()
vv_grid = vv.assign(x=vv.geometry.x, y=vv.geometry.y).pivot(index="y", columns="x", values="favorability").sort_index(ascending=False)
proxy_grid = probability.pivot(index="y", columns="x", values="probability").sort_index(ascending=False)
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), constrained_layout=True)
vv_image = axes[0].imshow(vv_grid.to_numpy(), extent=thermal_extent, origin="upper", cmap="viridis", vmin=0, vmax=5)
proxy_image = axes[1].imshow(proxy_grid.to_numpy(), extent=thermal_extent, origin="upper", cmap="magma", vmin=0, vmax=1)
thermal_image = axes[2].imshow(thermal_probability, extent=thermal_extent, origin="upper", cmap="magma", vmin=0, vmax=1)
fig.colorbar(vv_image, ax=axes[0], label="Ordinal score")
fig.colorbar(proxy_image, ax=axes[1], label="Probability")
fig.colorbar(thermal_image, ax=axes[2], label="Probability")
axes[0].set_title("Traditional VoterVeto heat score")
axes[1].set_title("GBLK P(proxy positive)")
axes[2].set_title("Stanford P(T at 3 km > 150 °C)")
for axis in axes:
    axis.set_axis_off()
plt.show()

In [ ]:
cv = run_gblk_calibration_cv(
    pfa,
    config,
    components=("heat",),
    n_folds=5,
    n_bins=5,
    random_state=0,
    nc=config.spatial_field.lattice_centers_per_dimension,
)["heat"]

def fold_ece(fold) -> float:
    reliability = fold.reliability
    populated = reliability.bin_counts > 0
    counts = reliability.bin_counts[populated]
    gaps = np.abs(
        reliability.bin_mean_pred[populated] - reliability.bin_fracs[populated]
    )
    return float(np.sum(counts * gaps) / np.sum(counts))

metric_distributions = {
    "brier_score": [fold.brier_score for fold in cv.folds],
    "brier_skill_score": [fold.brier_skill_score for fold in cv.folds],
    "expected_calibration_error": [fold_ece(fold) for fold in cv.folds],
}
fold_metrics = pd.DataFrame(
    {
        "fold": [fold.fold_id for fold in cv.folds],
        "n_train": [fold.extra["n_train"] for fold in cv.folds],
        "n_test": [fold.n_test for fold in cv.folds],
        "n_buffered": [fold.extra["n_buffered"] for fold in cv.folds],
        **metric_distributions,
    }
)
fold_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
axes[0].bar(fold_metrics.fold - 0.18, fold_metrics.brier_score, width=0.36, label="Brier score")
axes[0].bar(fold_metrics.fold + 0.18, fold_metrics.brier_skill_score, width=0.36, label="Brier skill")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(xlabel="Spatial fold", ylabel="Score", title="Replicate-level proper scores")
axes[0].legend()
axes[1].bar(fold_metrics.fold, fold_metrics.expected_calibration_error, color="#6A3D9A")
axes[1].set(xlabel="Spatial fold", ylabel="ECE", title="Replicate-level calibration error")
plt.show()
{
    "fold_metric_distribution": fold_metrics.describe().to_dict(),
    "thermal_probability_mean": float(np.nanmean(thermal_probability)),
    "thermal_probability_max": float(np.nanmax(thermal_probability)),
}

## Interpretation and caveat

The blocked scores validate discrimination and probability quality for the 145-site GDR1351 proxy design (83 positives and 62 designed pseudo-absences), not for direct temperature measurements or discovery of an economic resource. The 3 km thermal probability is independently defined physical context from a regional thermal model.